# Предсказание стоимости жилья

## Подготовка данных

### Инициализировать локальную Spark-сессию

In [2]:
import pandas as pd 
import numpy as np

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import pyspark.sql.functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.feature import Imputer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.mllib.evaluation import RegressionMetrics

pyspark_version = pyspark.__version__
if int(pyspark_version[:1]) == 3:
    from pyspark.ml.feature import OneHotEncoder    
elif int(pyspark_version[:1]) == 2:
    from pyspark.ml.feature import OneHotEncodeEstimator
        
RANDOM_SEED = 12345

In [3]:
spark = SparkSession.builder \
                    .master("local") \
                    .appName("Housing - LinearRegression") \
                    .getOrCreate()

### Прочитать содержимое файла

In [4]:
data = spark.read.option('header', 'true').csv('/datasets/housing.csv', inferSchema = True)
data.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)
 |-- ocean_proximity: string (nullable = true)



### Вывести типы данных колонок датасета. Использовать методы pySpark

In [5]:
print(pd.DataFrame(data.dtypes, columns=['column', 'type']))

               column    type
0           longitude  double
1            latitude  double
2  housing_median_age  double
3         total_rooms  double
4      total_bedrooms  double
5          population  double
6          households  double
7       median_income  double
8  median_house_value  double
9     ocean_proximity  string


**В колонках датасета содержатся следующие данные:**  

**количественные**

longitude — широта;  
latitude — долгота;  
housing_median_age — медианный возраст жителей жилого массива;  
total_rooms — общее количество комнат в домах жилого массива;  
total_bedrooms — общее количество спален в домах жилого массива;  
population — количество человек, которые проживают в жилом массиве;  
households — количество домовладений в жилом массиве;   
median_income — медианный доход жителей жилого массива;  
median_house_value — медианная стоимость дома в жилом массиве;  

**качественные**

ocean_proximity — близость к океану.  


## Предобработка данных

Проверим датасет на наличие пропусков

In [6]:
columns = data.columns

for column in columns:
    print(column, data.where(F.isnan(column) | F.col(column).isNull()).count())

longitude 0
latitude 0
housing_median_age 0
total_rooms 0
total_bedrooms 207
population 0
households 0
median_income 0
median_house_value 0
ocean_proximity 0


Колонка total_bedrooms содержит 207 пропусков (примерно 1% от общего количества записей).  
По условиям проекта, удалять пропуски нельзя, поэтому позже заменим их на медианные значения.  

Посмотрим описательные статистики с помощью метода describe()

In [9]:
data.describe().toPandas().T

,0,1,2,3,4
summary,count,mean,stddev,min,max
longitude,20640,-119.56970445736148,2.003531723502584,-124.35,-114.31
latitude,20640,35.6318614341087,2.135952397457101,32.54,41.95
housing_median_age,20640,28.639486434108527,12.58555761211163,1.0,52.0
total_rooms,20640,2635.7630813953488,2181.6152515827944,2.0,39320.0
total_bedrooms,20433,537.8705525375618,421.38507007403115,1.0,6445.0
population,20640,1425.4767441860465,1132.46212176534,3.0,35682.0
households,20640,499.5396802325581,382.3297528316098,1.0,6082.0
median_income,20640,3.8706710029070246,1.899821717945263,0.4999,15.0001
median_house_value,20640,206855.81690891474,115395.61587441359,14999.0,500001.0


Аномалий не обнаружено

**Разделим колонки на два типа: числовые и текстовые. Целевой признак отметим в переменной target**

In [10]:
categorical_cols = ['ocean_proximity']
numerical_cols  = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 
                   'population', 'households', 'median_income']
target = 'median_house_value' 

**Разделение на выборки**

Разделим датасет на две части: тренировочную и тестовую выборки

In [11]:
train_data, test_data = data.randomSplit([.8,.2], seed=RANDOM_SEED)
print(train_data.count(), test_data.count())

16431 4209


**Создадим трансформеры**

In [12]:
# Трансформируем категориальные признаки с помощью StringIndexer()
indexer = StringIndexer(inputCols=categorical_cols, outputCols=[c+'_idx' for c in categorical_cols])

# Преобразуем колонку с категориальными значениями техникой One hot encoding
encoder = OneHotEncoder(inputCols=[c+'_idx' for c in categorical_cols], 
                        outputCols=[c+'_ohe' for c in categorical_cols])

# Заменим пропуски на медианные значения с помощью Imputer
numerical_imputer = Imputer(strategy='median', inputCols=numerical_cols, 
                             outputCols=['imputed_'+col for col in numerical_cols])

# Преобразуем признаки в один вектор
categorical_assembler = VectorAssembler(inputCols=[c+'_ohe' for c in categorical_cols], 
                                        outputCol="categorical_features")

numerical_assembler = VectorAssembler(inputCols=['imputed_'+col for col in numerical_cols], 
                                      outputCol="numerical_features")

# Для числовых признаков тоже нужна трансформация — шкалирование значений, 
# чтобы сильные выбросы не смещали предсказания модели. Применим StandardScaler.
scaler = StandardScaler(inputCol='numerical_features', outputCol="numerical_features_scaled")

# Cоберем трансформированные категорийные и числовые признаки с помощью VectorAssembler
feature_assembler = VectorAssembler(inputCols=['categorical_features', 'numerical_features_scaled'], 
                                    outputCol='features', handleInvalid='skip')

### Вывод

На этапе Предобработки и Разведочного анализа данных:  

* Проверили датасет на наличие пропусков

* Проверили описательные статистики с помощью метода describe(). Аномалий не обнаружили

* Разделили датасет на две выборки: тестовую и тренировочную

* Разделили колонки на два типа: числовые и текстовые. Целевой признак сохранили в переменной target

* Трансформировали категориальные признаки с помощью StringIndexer()

* Преобразовали колонки с категориальными значениями техникой One hot encoding

* Заменили пропуски в колонке total_bedrooms (207 пропусков) на медианные значения с помощью метода Imputer

* Трансформировали числовые признаки с помощью StandardScaler.

* Cобрали трансформированные категорийные и числовые признаки с помощью VectorAssembler  

## Обучение моделей

Построим две модели линейной регрессии на разных наборах данных:  

- используя все данные из файла;  
- используя только числовые переменные, исключив категориальные.  

Для построения модели используем оценщик LinearRegression из библиотеки MLlib.  

In [13]:
# Модель линейной регрессии (LinearRegression) для всех данных из файла:
lr_all_features = LinearRegression(labelCol=target, featuresCol='features', 
                                   maxIter=10, regParam=0.3, elasticNetParam=0.8)

# Модель линейной регрессии (LinearRegression) только для числовых переменных:
lr_num_features = LinearRegression(labelCol=target, featuresCol='numerical_features_scaled', 
                                   maxIter=10, regParam=0.3, elasticNetParam=0.8)

Создадим Pipeline для первой модели линейной регрессии (со всеми данными), обучим на тренировочной выборке и предскажем результаты целевого признака на тестовой выборке

In [14]:
pipeline_all_features = Pipeline(stages=[indexer, encoder, numerical_imputer, categorical_assembler,
                                          numerical_assembler, scaler, feature_assembler, 
                                          lr_all_features])


model_all_features = pipeline_all_features.fit(train_data)

predictions_all = model_all_features.transform(test_data)

23/11/30 09:22:51 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
23/11/30 09:22:51 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS


In [15]:
# Выведем результаты предсказания
predictions_all.show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+-------------------+-------------------+---------------------+----------------------+-----------------+------------------+------------------+--------------------------+-------------------+----------------+--------------------+--------------------+-------------------------+--------------------+------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|ocean_proximity_idx|ocean_proximity_ohe|imputed_median_income|imputed_total_bedrooms|imputed_longitude|imputed_population|imputed_households|imputed_housing_median_age|imputed_total_rooms|imputed_latitude|categorical_features|  numerical_features|numerical_features_scaled|            features|        prediction|
+---------+--------+------------------+-----------+--------------+----------+----------+----------

Создадим Pipeline для второй модели линейной регрессии (с числовыми переменными), обучим на тренировочной выборке и предскажем результаты целевого признака на тестовой выборке

In [16]:
pipeline_num_features = Pipeline(stages=[numerical_imputer, numerical_assembler, scaler, lr_num_features])

model_num_features = pipeline_num_features.fit(train_data)
predictions_num = model_num_features.transform(test_data)

In [17]:
# Выведем результаты предсказания
predictions_num.show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------------+----------------------+-----------------+------------------+------------------+--------------------------+-------------------+----------------+--------------------+-------------------------+------------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|imputed_median_income|imputed_total_bedrooms|imputed_longitude|imputed_population|imputed_households|imputed_housing_median_age|imputed_total_rooms|imputed_latitude|  numerical_features|numerical_features_scaled|        prediction|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+---------------------+----------------------+-----------------+------------------+------------------+------------------------

### Вывод

На этапе обучения моделей построили две модели линейной регрессии (LinearRegression из библиотеки MLlib) на разных наборах данных:  

* используя все данные из файла;  
* используя только числовые переменные, исключив категориальные.  

Обучение провели на тренировочной выборке, а предсказание сделали на тестовой.

## Анализ результатов

Для оценки качества моделей будем использовать метрики: RMSE, MAE и R2.

In [18]:
# Оценим качество первой модели (для всех данных из файла)
evaluator_rmse_all = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="rmse")
evaluator_mae_all = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="mae")
evaluator_r2_all = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="r2")

rmse_all_features = evaluator_rmse_all.evaluate(predictions_all)
mae_all_features = evaluator_mae_all.evaluate(predictions_all)
r2_all_features = evaluator_r2_all.evaluate(predictions_all)

print("RMSE on all features: {:.2f}".format(rmse_all_features))
print("MAE on all features: {:.2f}".format(mae_all_features))
print("R2 on all features: {:.2f}".format(r2_all_features))

RMSE on all features: 68172.99
MAE on all features: 49290.64
R2 on all features: 0.65


In [19]:
# Оценим качество второй модели (только для числовых переменных)
evaluator_rmse_num = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="rmse")
evaluator_mae_num = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="mae")
evaluator_r2_num = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="r2")

rmse_num_features = evaluator_rmse_num.evaluate(predictions_num)
mae_num_features = evaluator_mae_num.evaluate(predictions_num)
r2_num_features = evaluator_r2_num.evaluate(predictions_num)

# Вывод результатов для числовых переменных
print("RMSE on numerical features: {:.2f}".format(rmse_num_features))
print("MAE on numerical features: {:.2f}".format(mae_num_features))
print("R2 on numerical features: {:.2f}".format(r2_num_features))

RMSE on numerical features: 69027.46
MAE on numerical features: 50285.14
R2 on numerical features: 0.65


Исходя из полученных результатов, метрики RMSE, MAE и R2 в обоих моделях не сильно отличаются. Чуть более точно работает первая модель, использующая все данные из файла.

In [20]:
# Исследование завершено, остановим сессию
spark.stop()

## Вывод

На этапе Обучения моделей:  

* Построили две модели линейной регрессии (LinearRegression из библиотеки MLlib) на разных наборах данных:  

 - используя все данные из файла;  
 - используя только числовые переменные, исключив категориальные.  

* Обучение провели на тренировочной выборке, а предсказание сделали на тестовой.  

На этапе Анализ результатов:

* Оценивали качество моделей, используя метрики: RMSE, MAE и R2. 
 
Исходя из полученных результатов, метрики RMSE, MAE и R2 в обоих моделях не сильно отличаются. Чуть более точно работает первая модель, использующая все данные из файла.  